# ROS Bag 交互式分析 Notebook

这个notebook允许你：
1. 一次性读取ROS bag数据
2. 交互式地分析和可视化数据  
3. 点击热力图查看特征点时间分布
4. 无需重复读取bag文件

## 1. 导入必要的库

In [ ]:
import rosbag
import numpy as np
import matplotlib.pyplot as plt
from geometry_msgs.msg import Point
from scipy.interpolate import interp1d
from scipy.spatial.transform import Rotation as R
import warnings
warnings.filterwarnings('ignore')

# 设置matplotlib为交互模式
%matplotlib inline

print("✓ 库导入成功")

## 2. 配置参数

In [ ]:
# 配置ROS bag文件路径
import os

# 指定包含.bag文件的目录
BAG_DIR = "/home/smoggy/workspace_ros1/r2d2/experiments/fuel/"

# 获取该目录下所有.bag文件的完整路径，并排序
BAG_FILES = sorted([
    os.path.join(BAG_DIR, f)
    for f in os.listdir(BAG_DIR)
    if f.endswith('.bag') and f.startswith('construction')
])

print(f"自动发现到 {len(BAG_FILES)} 个bag文件:")
for bf in BAG_FILES:
    print(" -", bf)
# BAG_FILE = "/home/smoggy/workspace_ros1/r2d2/experiments/vo_safe/test_simple_exp_3_210642.bag"
# BAG_FILE = "/home/smoggy/workspace_ros1/r2d2/experiments/fuel/test_simple_exp_3_212158.bag"
# BAG_FILE = "/home/smoggy/workspace_ros1/r2d2/experiments/fuel/test_simple_exp_4_212422.bag"

# 热力图参数
OV_WIDTH = 960
OV_HEIGHT = 540  
R2D2_WIDTH = 1280
R2D2_HEIGHT = 720
BIN_SIZE = 20

# 颜色范围
R2D2_SCALE = [0.0, 0.1]
OV_SCALE = [0.0, 1.0]

# print(f"配置完成: {BAG_FILE}")

## 3. 读取ROS Bag数据（只需运行一次）

In [ ]:
exploration_datas = []
r2d2_pc_datas = []
ov_pc_datas = []
gt_odom_datas = []
vins_odom_datas = []

for BAG_FILE in BAG_FILES:    
    print(f"正在读取ROS bag: {BAG_FILE}")

    exploration_data = []
    r2d2_pc_data = []
    ov_pc_data = []
    gt_odom_data = []
    vins_odom_data = []

    with rosbag.Bag(BAG_FILE, 'r') as bag:
        for topic, msg, t in bag.read_messages():
            timestamp = t.to_sec()
            
            if topic == '/exploration_rate':
                rate = msg.data if hasattr(msg, 'data') else float(msg)
                exploration_data.append([timestamp, rate])
            elif topic == '/r2d2/visible_features_uv':
                r2d2_pc_data.append([timestamp, msg])
            elif topic == '/ov_msckf/loop_feats':
                ov_pc_data.append([timestamp, msg])
            elif topic == '/kingfisher/ground_truth/odometry':
                gt_odom_data.append([timestamp, msg])
            elif topic == '/ov_msckf/loop_pose':
                vins_odom_data.append([timestamp, msg])

    print(f"✓ 找到 {len(exploration_data)} 条探索率消息")
    print(f"✓ 找到 {len(r2d2_pc_data)} 条R2D2点云消息")
    print(f"✓ 找到 {len(ov_pc_data)} 条OV-MSCKF特征消息")
    print(f"✓ 找到 {len(gt_odom_data)} 条地面真实里程计消息")
    print(f"✓ 找到 {len(vins_odom_data)} 条VINS里程计消息")

    exploration_datas.append(exploration_data)
    r2d2_pc_datas.append(r2d2_pc_data)
    ov_pc_datas.append(ov_pc_data)
    gt_odom_datas.append(gt_odom_data)
    vins_odom_datas.append(vins_odom_data)


## 4. 提取点云UV坐标和时间戳

In [ ]:
print("正在提取点云数据...")

# 为每个rosbag创建子图
num_bags = len(gt_odom_datas)
fig, axes = plt.subplots(num_bags, 2, figsize=(12, 4 * num_bags), squeeze=False)

all_starting_timestamps = []
all_durations = []

for idx, gt_odom_data in enumerate(gt_odom_datas):
    gt_timestamps = []
    gt_linear_vel = []
    gt_angular_vel = []
    starting_timestamp = None

    for timestamp, msg in gt_odom_data:
        gt_timestamps.append(timestamp)
        # Linear velocity components
        vx = msg.twist.twist.linear.x 
        vy = msg.twist.twist.linear.y
        vz = msg.twist.twist.linear.z
        linear_vel = np.sqrt(vx**2 + vy**2 + vz**2)  # Calculate magnitude
        if linear_vel > 0.1 and starting_timestamp is None:
            starting_timestamp = timestamp  # Relative to first timestamp

        # Angular velocity components
        wx = msg.twist.twist.angular.x
        wy = msg.twist.twist.angular.y
        wz = msg.twist.twist.angular.z
        angular_vel = np.sqrt(wx**2 + wy**2 + wz**2)  # Calculate magnitude

        gt_linear_vel.append(linear_vel)
        gt_angular_vel.append(angular_vel)

    # Convert to numpy arrays and relative time
    gt_timestamps = np.array(gt_timestamps)
    gt_linear_vel = np.array(gt_linear_vel)
    gt_angular_vel = np.array(gt_angular_vel)

    # Plot for this bag
    ax1 = axes[idx, 0]
    ax2 = axes[idx, 1]
    ax1.plot(gt_timestamps, gt_linear_vel, 'b-', label='Linear Velocity')
    ax1.grid(True)
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Linear Velocity (m/s)')
    ax1.legend()
    ax1.set_title(f'Bag {idx+1} Linear Velocity')

    ax2.plot(gt_timestamps, gt_angular_vel, 'r-', label='Angular Velocity')
    ax2.grid(True)
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Angular Velocity (rad/s)')
    ax2.legend()
    ax2.set_title(f'Bag {idx+1} Angular Velocity')

    if starting_timestamp is not None:
        print(f"[Bag {idx+1}] starting_timestamp:", starting_timestamp)
        print(f"[Bag {idx+1}] ending_timestamp:", gt_timestamps[-1])
        duration = gt_timestamps[-1] - starting_timestamp
        print(f"[Bag {idx+1}] ✓ 运动持续时间: {duration:.2f} 秒")
        all_starting_timestamps.append(starting_timestamp)
        all_durations.append(duration)
    else:
        print(f"[Bag {idx+1}] 未检测到速度大于0.1的起始时刻")
        all_starting_timestamps.append(None)
        all_durations.append(None)

plt.tight_layout()
plt.show()

In [ ]:
# 为每个rosbag创建子图，分别显示OV-MSCKF和R2D2特征热力图

import struct
import matplotlib.pyplot as plt

num_bags = len(r2d2_pc_datas) 
fig, axes = plt.subplots(num_bags, 2, figsize=(16, 4 * num_bags), squeeze=False)

for idx in range(num_bags):
    r2d2_pc_data = r2d2_pc_datas[idx]
    ov_pc_data = ov_pc_datas[idx]
    starting_timestamp = all_starting_timestamps[idx]
    duration = all_durations[idx]

    # 处理R2D2数据
    r2d2_u = []
    r2d2_v = []
    r2d2_intensity = []
    r2d2_time = []
    for timestamp, msg in r2d2_pc_data:
        if timestamp > starting_timestamp:
            msg_data = bytes(msg.data)
            point_step = msg.point_step
            num_points = msg.width
            for i in range(num_points):
                offset = i * point_step
                point_data = msg_data[offset:offset + point_step]
                u, v, intensity = struct.unpack('<fff', point_data)
                r2d2_u.append(u)
                r2d2_v.append(v)
                r2d2_intensity.append(intensity)
                r2d2_time.append(timestamp)

    # 处理OV-MSCKF数据
    ov_msckf_u_coords = []
    ov_msckf_v_coords = []
    ov_time = []
    highlight_uv_timestamps = []
    for timestamp, msg in ov_pc_data:
        if timestamp > starting_timestamp:
            for i in range(len(msg.channels)):
                u = msg.channels[i].values[2]
                v = msg.channels[i].values[3]
                ov_msckf_u_coords.append(u)
                ov_msckf_v_coords.append(v)
                ov_time.append(timestamp)
            if 760 < u < 780 and 280 < v < 300:
                highlight_uv_timestamps.append(timestamp)


    # 转换为numpy数组
    r2d2_u = np.array(r2d2_u)
    r2d2_v = np.array(r2d2_v)
    r2d2_intensity = np.array(r2d2_intensity)
    r2d2_time = np.array(r2d2_time)
    ov_u = np.array(ov_msckf_u_coords)
    ov_v = np.array(ov_msckf_v_coords)
    ov_time = np.array(ov_time)

    print(f"[Bag {idx+1}] ✓ 提取了 {len(r2d2_u)} 个R2D2特征点")
    print(f"[Bag {idx+1}] ✓ 提取了 {len(ov_u)} 个OV-MSCKF特征点")
    if len(r2d2_u) > 0:
        print(f"  R2D2 UV范围: u=[{r2d2_u.min():.1f}, {r2d2_u.max():.1f}], v=[{r2d2_v.min():.1f}, {r2d2_v.max():.1f}]")

    # OV-MSCKF热力图
    ax_ov = axes[idx, 0]
    if len(ov_u) > 0 and duration is not None and duration > 0:
        heatmap_ov, xedges_ov, yedges_ov = np.histogram2d(
            ov_u, ov_v,
            bins=[OV_WIDTH // BIN_SIZE, OV_HEIGHT // BIN_SIZE],
            range=[[0, OV_WIDTH], [0, OV_HEIGHT]]
        )
        heatmap_ov = heatmap_ov / duration
        im_ov = ax_ov.imshow(
            heatmap_ov.T,
            origin='lower',
            extent=[xedges_ov[0], xedges_ov[-1], yedges_ov[0], yedges_ov[-1]],
            aspect='auto',
            cmap='viridis',
            vmin=0,
            vmax=1.5
        )
        plt.colorbar(im_ov, ax=ax_ov, label='OV-MSCKF feature density (points/s)')
    ax_ov.set_title(f'Bag {idx+1} OV-MSCKF feature heatmap')
    ax_ov.set_xlabel('u (pixels)')
    ax_ov.set_ylabel('v (pixels)')
    ax_ov.grid(False)

    # R2D2热力图
    ax_r2d2 = axes[idx, 1]
    if len(r2d2_u) > 0 and duration is not None and duration > 0:
        heatmap_r2d2, xedges_r2d2, yedges_r2d2 = np.histogram2d(
            r2d2_u, r2d2_v,
            bins=[R2D2_WIDTH // BIN_SIZE, R2D2_HEIGHT // BIN_SIZE],
            range=[[0, R2D2_WIDTH], [0, R2D2_HEIGHT]],
            weights=r2d2_intensity
        )
        heatmap_r2d2 = heatmap_r2d2 / duration
        im_r2d2 = ax_r2d2.imshow(
            heatmap_r2d2.T,
            origin='lower',
            extent=[xedges_r2d2[0], xedges_r2d2[-1], yedges_r2d2[0], yedges_r2d2[-1]],
            aspect='auto',
            cmap='viridis',
            vmin=0,
            vmax=0.13
        )
        plt.colorbar(im_r2d2, ax=ax_r2d2, label='r2d2 feature density (points/s)')
    ax_r2d2.set_title(f'Bag {idx+1} r2d2 feature heatmap')
    ax_r2d2.set_xlabel('u (pixels)')
    ax_r2d2.set_ylabel('v (pixels)')
    ax_r2d2.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
print("计算并绘制里程计误差...")

# 辅助函数（参考 batch_analysis.ipynb ）
def calculate_position_error(gt_pos, est_pos) -> float:
    dx = gt_pos.x - est_pos.x
    dy = gt_pos.y - est_pos.y
    dz = gt_pos.z - est_pos.z
    return float(np.sqrt(dx*dx + dy*dy + dz*dz))


def calculate_orientation_error(gt_quat, est_quat) -> float:
    if hasattr(gt_quat, 'x'):
        gt_quat_array = [gt_quat.x, gt_quat.y, gt_quat.z, gt_quat.w]
    else:
        gt_quat_array = gt_quat
    if hasattr(est_quat, 'x'):
        est_quat_array = [est_quat.x, est_quat.y, est_quat.z, est_quat.w]
    else:
        est_quat_array = est_quat
    r_gt = R.from_quat(gt_quat_array)
    r_est = R.from_quat(est_quat_array)
    r_rel = r_gt * r_est.inv()
    angle = r_rel.magnitude()
    return float(angle)


def compute_rigid_alignment_from_first_odoms(gt_first_msg, vins_first_msg):
    gt_initial_pos = gt_first_msg.pose.pose.position
    vins_initial_pos = vins_first_msg.pose.pose.position
    gt_initial_quat = gt_first_msg.pose.pose.orientation
    vins_initial_quat = vins_first_msg.pose.pose.orientation

    r_gt_initial = R.from_quat([gt_initial_quat.x, gt_initial_quat.y, gt_initial_quat.z, gt_initial_quat.w])
    r_vins_initial = R.from_quat([vins_initial_quat.x, vins_initial_quat.y, vins_initial_quat.z, vins_initial_quat.w])
    r_transform = r_gt_initial * r_vins_initial.inv()

    gt_pos_array = np.array([gt_initial_pos.x, gt_initial_pos.y, gt_initial_pos.z], dtype=float)
    vins_pos_array = np.array([vins_initial_pos.x, vins_initial_pos.y, vins_initial_pos.z], dtype=float)
    vins_pos_rotated = r_transform.apply(vins_pos_array)
    translation = gt_pos_array - vins_pos_rotated
    return r_transform, translation


# 逐bag计算误差并绘制
num_bags = len(gt_odom_datas)
fig, axes = plt.subplots(num_bags, 2, figsize=(16, 4 * num_bags), squeeze=False)

for idx in range(num_bags):
    gt_odom_data = gt_odom_datas[idx]
    vins_odom_data = vins_odom_datas[idx] if idx < len(vins_odom_datas) else []
    start_time = all_starting_timestamps[idx]

    ax_pos = axes[idx, 0]
    ax_ori = axes[idx, 1]

    if start_time is None or len(gt_odom_data) == 0 or len(vins_odom_data) == 0:
        ax_pos.set_title(f'Bag {idx+1} 缺少有效数据，无法计算误差')
        ax_pos.axis('off')
        ax_ori.axis('off')
        print(f"[Bag {idx+1}] 缺少数据，跳过误差计算")
        continue

    gt_times = np.array([t for t, _ in gt_odom_data], dtype=float)
    vins_times = np.array([t for t, _ in vins_odom_data], dtype=float)

    gt_mask = gt_times >= start_time
    vins_mask = vins_times >= start_time
    if not np.any(gt_mask) or not np.any(vins_mask):
        ax_pos.set_title(f'Bag {idx+1} 无重叠时间段')
        ax_pos.axis('off')
        ax_ori.axis('off')
        print(f"[Bag {idx+1}] 无重叠时间段，跳过")
        continue

    first_common_time = max(gt_times[gt_mask][0], vins_times[vins_mask][0])
    gt_first_idx = int(np.argmin(np.abs(gt_times - first_common_time)))
    vins_first_idx = int(np.argmin(np.abs(vins_times - first_common_time)))

    gt_first_msg = gt_odom_data[gt_first_idx][1]
    vins_first_msg = vins_odom_data[vins_first_idx][1]

    r_transform, translation = compute_rigid_alignment_from_first_odoms(gt_first_msg, vins_first_msg)

    error_times = []
    position_errors = []
    orientation_errors = []

    end_time = min(gt_times[-1], vins_times[-1])

    for vins_time, vins_msg in vins_odom_data:
        if vins_time < first_common_time or vins_time > end_time:
            continue
        gt_diffs = np.abs(gt_times - vins_time)
        gt_idx = int(np.argmin(gt_diffs))
        if gt_diffs[gt_idx] > 0.1:
            continue
        gt_msg = gt_odom_data[gt_idx][1]

        vins_pos_array = np.array([
            vins_msg.pose.pose.position.x,
            vins_msg.pose.pose.position.y,
            vins_msg.pose.pose.position.z,
        ])
        vins_pos_rotated = r_transform.apply(vins_pos_array)
        vins_pos_transformed = vins_pos_rotated + translation

        aligned_vins_pos = Point()
        aligned_vins_pos.x = float(vins_pos_transformed[0])
        aligned_vins_pos.y = float(vins_pos_transformed[1])
        aligned_vins_pos.z = float(vins_pos_transformed[2])

        r_vins_current = R.from_quat([
            vins_msg.pose.pose.orientation.x,
            vins_msg.pose.pose.orientation.y,
            vins_msg.pose.pose.orientation.z,
            vins_msg.pose.pose.orientation.w,
        ])
        r_vins_aligned = r_transform * r_vins_current

        pos_error = calculate_position_error(gt_msg.pose.pose.position, aligned_vins_pos)
        orient_error = calculate_orientation_error(
            gt_msg.pose.pose.orientation,
            r_vins_aligned.as_quat(),
        )

        error_times.append(vins_time - first_common_time)
        position_errors.append(pos_error)
        orientation_errors.append(orient_error)

    if len(position_errors) > 0:
        error_times = np.array(error_times)
        position_errors = np.array(position_errors)
        orientation_errors = np.array(orientation_errors)

        ax_pos.plot(error_times, position_errors, 'b-', label='Position Error (m)')
        ax_pos.set_title(f'Bag {idx+1} 位置误差')
        ax_pos.set_xlabel('Time since start (s)')
        ax_pos.set_ylabel('Error (m)')
        ax_pos.set_ylim(0, 5)
        ax_pos.grid(True)
        ax_pos.legend()

        ax_ori.plot(error_times, orientation_errors, 'r-', label='Orientation Error (rad)')
        ax_ori.set_title(f'Bag {idx+1} 姿态误差')
        ax_ori.set_xlabel('Time since start (s)')
        ax_ori.set_ylabel('Error (rad)')
        ax_ori.grid(True)
        ax_ori.legend()
    else:
        ax_pos.set_title(f'Bag {idx+1} 无有效误差数据')
        ax_pos.axis('off')
        ax_ori.axis('off')
        print(f"[Bag {idx+1}] 无有效误差数据")

plt.tight_layout()
plt.show()


In [ ]:
# 将对齐后的里程计与两条轨迹写入新的 rosbag（backup/aligned_*.bag）
import os
import numpy as np
import rosbag
from copy import deepcopy
from nav_msgs.msg import Odometry, Path
from geometry_msgs.msg import PoseStamped, Point
from scipy.spatial.transform import Rotation as R

# 目标输出目录（与当前工作目录相关联，可按需修改为每个bag所在目录下的 backup）
BACKUP_DIR = os.path.abspath('backup')
os.makedirs(BACKUP_DIR, exist_ok=True)

# 安全获取 frame_id

def _get_frame_id(msg, default_frame='map'):
    try:
        fid = str(msg.header.frame_id)
        return fid if fid else default_frame
    except Exception:
        return default_frame

# 将位姿写入 PoseStamped

def _make_pose_stamped(stamp, frame_id, position_xyz, quat_wxyz):
    ps = PoseStamped()
    ps.header.stamp = stamp
    ps.header.frame_id = frame_id
    ps.pose.position.x = float(position_xyz[0])
    ps.pose.position.y = float(position_xyz[1])
    ps.pose.position.z = float(position_xyz[2])
    # 输入为 (w, x, y, z) 或 (x, y, z, w)? 这里统一使用 scipy 返回的 [x,y,z,w]
    ps.pose.orientation.x = float(quat_wxyz[0])
    ps.pose.orientation.y = float(quat_wxyz[1])
    ps.pose.orientation.z = float(quat_wxyz[2])
    ps.pose.orientation.w = float(quat_wxyz[3])
    return ps

# 针对每个 bag 重新计算配准并写出新的 bag
num_bags = len(gt_odom_datas)
for idx in range(num_bags):
    gt_odom_data = gt_odom_datas[idx]
    vins_odom_data = vins_odom_datas[idx] if idx < len(vins_odom_datas) else []
    start_time = all_starting_timestamps[idx]

    if start_time is None or len(gt_odom_data) == 0 or len(vins_odom_data) == 0:
        print(f"[Bag {idx+1}] 缺少数据，跳过导出")
        continue

    # 统一时间窗与首帧对齐
    gt_times = np.array([t for t, _ in gt_odom_data], dtype=float)
    vins_times = np.array([t for t, _ in vins_odom_data], dtype=float)
    gt_mask = gt_times >= start_time
    vins_mask = vins_times >= start_time
    if not np.any(gt_mask) or not np.any(vins_mask):
        print(f"[Bag {idx+1}] 无重叠时间段，跳过导出")
        continue

    first_common_time = max(gt_times[gt_mask][0], vins_times[vins_mask][0])
    gt_first_idx = int(np.argmin(np.abs(gt_times - first_common_time)))
    vins_first_idx = int(np.argmin(np.abs(vins_times - first_common_time)))

    gt_first_msg = gt_odom_data[gt_first_idx][1]
    vins_first_msg = vins_odom_data[vins_first_idx][1]

    # 刚性配准（与上面绘图单元一致）
    gt_initial_pos = gt_first_msg.pose.pose.position
    vins_initial_pos = vins_first_msg.pose.pose.position
    gt_initial_quat = gt_first_msg.pose.pose.orientation
    vins_initial_quat = vins_first_msg.pose.pose.orientation

    r_gt_initial = R.from_quat([gt_initial_quat.x, gt_initial_quat.y, gt_initial_quat.z, gt_initial_quat.w])
    r_vins_initial = R.from_quat([vins_initial_quat.x, vins_initial_quat.y, vins_initial_quat.z, vins_initial_quat.w])
    r_transform = r_gt_initial * r_vins_initial.inv()

    gt_pos_array = np.array([gt_initial_pos.x, gt_initial_pos.y, gt_initial_pos.z], dtype=float)
    vins_pos_array = np.array([vins_initial_pos.x, vins_initial_pos.y, vins_initial_pos.z], dtype=float)
    vins_pos_rotated = r_transform.apply(vins_pos_array)
    translation = gt_pos_array - vins_pos_rotated

    # 生成对齐后的里程计序列与两条路径
    end_time = min(gt_times[-1], vins_times[-1])
    FRAME_ID = 'world'
    aligned_path = Path()
    aligned_path.header.frame_id = FRAME_ID
    gt_path = Path()
    gt_path.header.frame_id = FRAME_ID

    aligned_odom_msgs = []

    for vins_time, vins_msg in vins_odom_data:
        if vins_time < first_common_time or vins_time > end_time:
            continue
        # 匹配最近的 GT
        gt_diffs = np.abs(gt_times - vins_time)
        gt_idx = int(np.argmin(gt_diffs))
        if gt_diffs[gt_idx] > 0.1:
            continue
        gt_msg = gt_odom_data[gt_idx][1]

        # 位姿对齐
        vins_pos = np.array([
            vins_msg.pose.pose.position.x,
            vins_msg.pose.pose.position.y,
            vins_msg.pose.pose.position.z,
        ], dtype=float)
        vins_pos_rot = r_transform.apply(vins_pos)
        vins_pos_aligned = vins_pos_rot + translation

        r_vins_cur = R.from_quat([
            vins_msg.pose.pose.orientation.x,
            vins_msg.pose.pose.orientation.y,
            vins_msg.pose.pose.orientation.z,
            vins_msg.pose.pose.orientation.w,
        ])
        r_vins_aligned = r_transform * r_vins_cur
        quat_xyzw = r_vins_aligned.as_quat()  # [x,y,z,w]

        # 构造对齐后的 Odometry（复制原消息，更新 header/pose）
        odom_aligned = Odometry()
        try:
            # 尽量保留原子段
            odom_aligned = deepcopy(vins_msg)
        except Exception:
            pass
        odom_aligned.header.stamp = vins_msg.header.stamp
        odom_aligned.header.frame_id = FRAME_ID
        odom_aligned.pose.pose.position.x = float(vins_pos_aligned[0])
        odom_aligned.pose.pose.position.y = float(vins_pos_aligned[1])
        odom_aligned.pose.pose.position.z = float(vins_pos_aligned[2])
        odom_aligned.pose.pose.orientation.x = float(quat_xyzw[0])
        odom_aligned.pose.pose.orientation.y = float(quat_xyzw[1])
        odom_aligned.pose.pose.orientation.z = float(quat_xyzw[2])
        odom_aligned.pose.pose.orientation.w = float(quat_xyzw[3])
        aligned_odom_msgs.append((vins_time, odom_aligned))

        # 轨迹点（aligned）
        aligned_ps = _make_pose_stamped(vins_msg.header.stamp, FRAME_ID, vins_pos_aligned, quat_xyzw)
        aligned_path.poses.append(aligned_ps)

        # 轨迹点（GT）
        gt_quat = [
            gt_msg.pose.pose.orientation.x,
            gt_msg.pose.pose.orientation.y,
            gt_msg.pose.pose.orientation.z,
            gt_msg.pose.pose.orientation.w,
        ]
        gt_ps = _make_pose_stamped(gt_msg.header.stamp, FRAME_ID,
                                   [gt_msg.pose.pose.position.x, gt_msg.pose.pose.position.y, gt_msg.pose.pose.position.z],
                                   gt_quat)
        gt_path.poses.append(gt_ps)

    if not aligned_odom_msgs:
        print(f"[Bag {idx+1}] 无可写入的对齐里程计，跳过导出")
        continue

    # 设置 Path header 时间戳（使用末尾时间）
    last_stamp = aligned_odom_msgs[-1][1].header.stamp
    aligned_path.header.stamp = last_stamp
    gt_path.header.stamp = last_stamp

    # 生成输出文件名
    out_name = f'aligned_{idx+1:02d}.bag'
    out_path = os.path.join(BACKUP_DIR, out_name)

    # 写入新 bag：包含对齐后的里程计，以及“随时间递增”的 Path
    with rosbag.Bag(out_path, 'w') as out_bag:
        # 1) 对齐后的里程计序列（与原时间戳一致）
        for t_sec, odom_msg in aligned_odom_msgs:
            out_bag.write('/aligned_odometry', odom_msg, odom_msg.header.stamp)

        # 2) 递增写出 Path：在每个时间点写入“当前累计”的路径，便于 RViz 实时显示
        # aligned_path
        for i in range(len(aligned_path.poses)):
            p = Path()
            p.header.frame_id = FRAME_ID
            p.header.stamp = aligned_path.poses[i].header.stamp
            p.poses = aligned_path.poses[:i+1]
            out_bag.write('/aligned_path', p, p.header.stamp)
        # gt_path
        for i in range(len(gt_path.poses)):
            p = Path()
            p.header.frame_id = FRAME_ID
            p.header.stamp = gt_path.poses[i].header.stamp
            p.poses = gt_path.poses[:i+1]
            out_bag.write('/gt_path', p, p.header.stamp)

    print(f"[Bag {idx+1}] 已写入: {out_path}  (topics: /aligned_odometry, /aligned_path, /gt_path)")



In [ ]:
# 修正版合并函数：避免多行 with 的缩进问题
import rosbag

def merge_two_bags_fixed(original_path: str, aligned_path: str, out_path: str):
    """按时间戳双路归并，写出包含两侧所有消息的新 bag（修正多行 with）。"""
    def _next(it):
        try:
            return next(it)
        except StopIteration:
            return None

    with rosbag.Bag(out_path, 'w') as out_bag:
        with rosbag.Bag(original_path, 'r') as bag_orig:
            with rosbag.Bag(aligned_path, 'r') as bag_aln:
                it_orig = bag_orig.read_messages()   # (topic, msg, t)
                it_aln  = bag_aln.read_messages()
                cur_o = _next(it_orig)
                cur_a = _next(it_aln)

                while cur_o is not None or cur_a is not None:
                    if cur_a is None:
                        topic, msg, t = cur_o
                        out_bag.write(topic, msg, t)
                        cur_o = _next(it_orig)
                        continue
                    if cur_o is None:
                        topic, msg, t = cur_a
                        out_bag.write(topic, msg, t)
                        cur_a = _next(it_aln)
                        continue
                    # 两侧都存在，比较时间戳
                    to, mo, to_t = cur_o
                    ta, ma, ta_t = cur_a
                    if float(to_t.to_sec()) <= float(ta_t.to_sec()):
                        out_bag.write(to, mo, to_t)
                        cur_o = _next(it_orig)
                    else:
                        out_bag.write(ta, ma, ta_t)
                        cur_a = _next(it_aln)



In [ ]:
# 使用修正版函数执行合并（写入 backup/merged_XX.bag）
import os

BACKUP_DIR = os.path.abspath('backup')
num_ok2 = 0
for idx, orig in enumerate(BAG_FILES):
    aligned_name = f'aligned_{idx+1:02d}.bag'
    aligned_path = os.path.join(BACKUP_DIR, aligned_name)
    if not os.path.isfile(aligned_path):
        print(f"[Merge2 {idx+1}] 缺少对齐 bag: {aligned_path}，跳过")
        continue
    if not os.path.isfile(orig):
        print(f"[Merge2 {idx+1}] 原始 bag 不存在: {orig}，跳过")
        continue
    out_path = os.path.join(BACKUP_DIR, f'merged_{idx+1:02d}.bag')
    try:
        merge_two_bags_fixed(orig, aligned_path, out_path)
        num_ok2 += 1
        print(f"[Merge2 {idx+1}] 已生成: {out_path}")
    except Exception as e:
        print(f"[Merge2 {idx+1}] 合并失败: {e}")

print(f"Done2. 成功合并 {num_ok2}/{len(BAG_FILES)} 个 bag，输出在: {BACKUP_DIR}")
